## Reading data for England 

In [1]:
import pandas as pd

df = pd.read_csv('../Extacted data/Pharm stats /Pharm_England_2021_2025.csv')
df.head(2)

,YEAR_MONTH,REGIONAL_OFFICE_NAME,REGIONAL_OFFICE_CODE,BNF_CHEMICAL_SUBSTANCE,CHEMICAL_SUBSTANCE_BNF_DESCR,QUANTITY,ITEMS,TOTAL_QUANTITY,ADQUSAGE,NIC,ACTUAL_COST
0,202101,EAST OF ENGLAND,Y61,0401010AC,Sodium oxybate,9720.0,23.0,11340.0,0.0,22680.00,21169.66950
1,202101,EAST OF ENGLAND,Y61,0401010AD,Melatonin,432989.0,13612.0,848532.0,0.0,507959.54,473801.90184


In [2]:
# have null value in column 'CHEMICAL_SUBSTANCE_BNF_DESCR'
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 97795 entries, 0 to 97794
Data columns (total 11 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   YEAR_MONTH                    97795 non-null  int64  
 1   REGIONAL_OFFICE_NAME          97795 non-null  str    
 2   REGIONAL_OFFICE_CODE          97795 non-null  str    
 3   BNF_CHEMICAL_SUBSTANCE        97795 non-null  str    
 4   CHEMICAL_SUBSTANCE_BNF_DESCR  81951 non-null  str    
 5   QUANTITY                      97795 non-null  float64
 6   ITEMS                         97795 non-null  float64
 7   TOTAL_QUANTITY                97795 non-null  float64
 8   ADQUSAGE                      97795 non-null  float64
 9   NIC                           97795 non-null  float64
 10  ACTUAL_COST                   97795 non-null  float64
dtypes: float64(6), int64(1), str(4)
memory usage: 12.0 MB


## Fill in nulls in CHEMICAL_SUBSTANCE_BNF_DESCR column from BNF_CHEMICAL_SUBSTANCE

In [3]:
df['CHEMICAL_SUBSTANCE_BNF_DESCR'] = df['CHEMICAL_SUBSTANCE_BNF_DESCR'].fillna(df['BNF_CHEMICAL_SUBSTANCE'])

## Dropping columns 

In [4]:
df = df.drop(columns=['REGIONAL_OFFICE_CODE','BNF_CHEMICAL_SUBSTANCE','TOTAL_QUANTITY','QUANTITY','ADQUSAGE'])

In [5]:
df.head(2)

,YEAR_MONTH,REGIONAL_OFFICE_NAME,CHEMICAL_SUBSTANCE_BNF_DESCR,ITEMS,NIC,ACTUAL_COST
0,202101,EAST OF ENGLAND,Sodium oxybate,23.0,22680.00,21169.66950
1,202101,EAST OF ENGLAND,Melatonin,13612.0,507959.54,473801.90184


## Changing type and names of columns

In [6]:
df['YEAR_MONTH'] = pd.to_datetime(df['YEAR_MONTH'], format='%Y%m').dt.strftime('%Y-%m')

In [7]:
df = df.rename(columns={
    'YEAR_MONTH': 'date',
    'REGIONAL_OFFICE_NAME': 'region',
    'CHEMICAL_SUBSTANCE_BNF_DESCR': 'chemical_substance',
    'ITEMS': 'items',
    'NIC': 'net_cost ',
    'ACTUAL_COST': 'actual_cost'
})

In [8]:
df.head(2)

,date,region,chemical_substance,items,net_cost,actual_cost
0,2021-01,EAST OF ENGLAND,Sodium oxybate,23.0,22680.00,21169.66950
1,2021-01,EAST OF ENGLAND,Melatonin,13612.0,507959.54,473801.90184


## Filtering antidepressants from the list

In [9]:
df['chemical_substance'].nunique()

264

In [10]:
# for item in sorted(df['chemical_substance'].unique()):
#     print(item)

In [11]:
antidepressants = [
    'Agomelatine', 'Amitriptyline hydrochloride', 
    'Bupropion hydrochloride', 'Citalopram hydrobromide', 'Citalopram hydrochloride',
    'Clomipramine hydrochloride', 'Dosulepin hydrochloride', 'Doxepin',
    'Duloxetine hydrochloride', 'Escitalopram', 'Fluoxetine hydrochloride', 'Fluvoxamine maleate',
    'Imipramine hydrochloride', 'Isocarboxazid', 'Lofepramine hydrochloride',
    'Mianserin hydrochloride', 'Mirtazapine', 'Moclobemide', 'Nortriptyline',
    'Paroxetine hydrochloride', 'Phenelzine sulfate', 'Reboxetine',
    'Sertraline hydrochloride', 'Tranylcypromine sulfate', 'Trazodone hydrochloride',
    'Trimipramine maleate', 'Venlafaxine', 'Vortioxetine'
]

anxiolytics = [
    'Alprazolam', 'Amobarbital', 'Amobarbital sodium', 'Bromazepam',
    'Buspirone hydrochloride', 'Butobarbital', 'Chloral hydrate',
    'Chlordiazepoxide hydrochloride', 'Clobazam', 'Clomethiazole',
    'Clomethiazole edisilate', 'Clonazepam', 'Cloral betaine', 'Daridorexant',
    'Diazepam', 'Eszopiclone', 'Flurazepam hydrochloride', 'Loprazolam mesilate',
    'Lorazepam', 'Lormetazepam', 'Meprobamate', 'Melatonin',
    'Midazolam hydrochloride', 'Midazolam maleate', 'Nitrazepam', 'Oxazepam',
    'Paraldehyde', 'Phenobarbital', 'Phenobarbital sodium', 'Pregabalin',
    'Promethazine teoclate', 'Secobarbital sodium', 'Sodium oxybate',
    'Temazepam', 'Zolpidem tartrate', 'Zopiclone'
]

# filter both groups
df_filtered = df[df['chemical_substance'].isin(antidepressants + anxiolytics)].copy()

# create new group column
df_filtered['group'] = df_filtered['chemical_substance'].apply(
    lambda x: 'Antidepressants' if x in antidepressants else 'Anxiolytics'
)

print(df_filtered.shape)
print(df_filtered['group'].value_counts())
print(len(antidepressants))
print(len(anxiolytics))

(24903, 7)
group
Antidepressants    12735
Anxiolytics        12168
Name: count, dtype: int64
28
36


In [12]:
df_filtered.tail(2)

,date,region,chemical_substance,items,net_cost,actual_cost,group
97792,2025-12,UNIDENTIFIED,Zolpidem tartrate,8.0,3.56,3.64598,Anxiolytics
97793,2025-12,UNIDENTIFIED,Zopiclone,54.0,18.92,20.81509,Anxiolytics


In [13]:
df_filtered['region'] = 'England'
df_filtered['items_1000'] = df_filtered['items']/1000
df_filtered = df_filtered.rename(columns={'region': 'country'})

In [14]:
# reorder columns
df_filtered = df_filtered[["date","country","group","chemical_substance","items"]]
df_filtered.head()

,date,country,group,chemical_substance,items
0,2021-01,England,Anxiolytics,Sodium oxybate,23.0
1,2021-01,England,Anxiolytics,Melatonin,13612.0
2,2021-01,England,Anxiolytics,Chloral hydrate,105.0
3,2021-01,England,Anxiolytics,Cloral betaine,5.0
4,2021-01,England,Anxiolytics,Clomethiazole edisilate,1.0


In [15]:
# save to csv
df_filtered.to_csv("EDA_England_pharm.csv", index=False)

### England Pharm per city

In [33]:
df_city = pd.read_csv('../Extacted data/Pharm stats /England_per_city_2021_2025.csv')
df_city.head(2)

,YEAR_MONTH,CITY,SUBSTANCE,CLASS,ITEMS,ACTUAL_COST
0,202101,Birmingham,Agomelatine,antidepressant,10.0,477.1802
1,202101,Birmingham,Amitriptyline hydrochloride,antidepressant,18253.0,37451.0438


In [34]:
df_city['YEAR_MONTH'] = pd.to_datetime(df_city['YEAR_MONTH'], format='%Y%m').dt.strftime('%Y-%m')
df_city['country'] = 'England'

In [35]:
df_city = df_city.rename(columns={
    'YEAR_MONTH': 'date',
    'CITY': 'city',
    'SUBSTANCE': 'substance',
    'ITEMS': 'items',
    'CLASS': 'group',
    'ACTUAL_COST': 'actual_cost'
})

In [36]:
df_city = df_city[["date","country","city","group","substance","items"]]
df_city.head()


,date,country,city,group,substance,items
0,2021-01,England,Birmingham,antidepressant,Agomelatine,10.0
1,2021-01,England,Birmingham,antidepressant,Amitriptyline hydrochloride,18253.0
2,2021-01,England,Birmingham,anxiolytic,Amobarbital sodium,1.0
3,2021-01,England,Birmingham,antidepressant,Bupropion hydrochloride,12.0
4,2021-01,England,Birmingham,anxiolytic,Buspirone hydrochloride,214.0


In [37]:
df_city.to_csv("EDA_England_pharm_per_city.csv", index=False)